# MLOps Internal Laboratory Examination
## Question 4: Develop and Automate an E-Commerce Product Return Prediction System using CI/CD

**Student Name:** Velpuri Bhanu Teja  
**Register Number:** 231FA04E85  
**Section:** Section 7  
**GitHub Repository:** https://github.com/BhanuTeja33/ecommerce-return-prediction-ci-cd  
**Container Image:** ghcr.io/bhanuteja33/ecommerce-return-prediction:latest  


### 1. Load Dataset & Initial Exploration


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/ecommerce_returns.csv')
print('Dataset Shape:', df.shape)
df.head()

### 2. Data Preprocessing: Handling Missing & Invalid Values


In [2]:
from src.data_preprocessing import clean_invalid_values, handle_missing_values

print('Missing values before preprocessing:')
print(df.isnull().sum()[df.isnull().sum() > 0])

cleaned_df, stats = handle_missing_values(df)
print('\nMissing values after preprocessing:')
print(cleaned_df.isnull().sum().sum())
print('\nComputed Baseline Imputation Statistics:', stats)

### 3. Feature Engineering
- `delivery_delay`: Actual_Delivery_Days - Expected_Delivery_Days
- `customer_return_frequency`: Previous_Returns / Previous_Orders
- `effective_price`: Product_Price_INR * (1 - Discount_Percent / 100)
- `is_delayed`: Indicator flag for delivery delay
- `return_risk_score`: Weighted risk indicator


In [3]:
from src.feature_engineering import engineer_features

engineered_df = engineer_features(cleaned_df)
engineered_df[['Order_ID', 'delivery_delay', 'customer_return_frequency', 'effective_price', 'is_delayed', 'return_risk_score']].head()

### 4. Train and Compare Classification Models


In [4]:
from src.train import train_models

best_pipeline, metrics = train_models()
for model_name, m in metrics.items():
    print(f'=== {model_name} ===')
    print(f"Accuracy : {m['accuracy']:.4f} | Recall: {m['recall']:.4f} | F1: {m['f1_score']:.4f} | ROC-AUC: {m['roc_auc']:.4f}")

### 5. Quality Gate & Model Inference Verification


In [5]:
from quality_gate import run_quality_gate
passed = run_quality_gate()
print('Quality Gate Passed:', passed)

### 6. FastAPI Local Test & Inference


In [6]:
import json
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)
health_res = client.get('/health')
print('Health Check:', health_res.json())

sample = {
    'Order_ID': 'ECOM00004',
    'Product_Category': 'Electronics',
    'Product_Price_INR': 22055.97,
    'Discount_Percent': 21.6,
    'Expected_Delivery_Days': 9,
    'Actual_Delivery_Days': 12,
    'Customer_Rating_PreOrder': 4.4,
    'Previous_Returns': 4,
    'Previous_Orders': 12,
    'Payment_Method': 'Card',
    'Damaged_Package_Flag': 'Yes',
    'Package_Size': 'Large',
    'Promotion_Active': 'Yes'
}
pred_res = client.post('/predict', json=sample)
print('Prediction:', json.dumps(pred_res.json(), indent=2))